# Part 2: Supervised Machine Learning Model - Build, Train, and Evaluate
This notebook splits features and labels, applies categorical encodings, scales inputs, trains Linear/Ridge regression, trains Logistic regression with class weights, and tests predictions at multiple thresholds.

### Load Dataset & Define Labels

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)

dataframe = pd.read_csv("cleaned_data.csv" if os.path.exists("cleaned_data.csv") else "part1/cleaned_data.csv")
features = dataframe.drop(columns=['customerID', 'ReferralCode', 'MonthlyCharges', 'Churn'])
target_regression = dataframe['MonthlyCharges']

target_classification = []
for val in dataframe['Churn']:
    if val == 'Yes':
        target_classification.append(1)
    else:
        target_classification.append(0)
target_classification = pd.Series(target_classification)
print("Features shape: " + str(features.shape))

### Encoding Categorical Features

In [ ]:
encoded_features = features.copy()
contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
encoded_features['Contract'] = encoded_features['Contract'].map(contract_mapping)

internet_mapping = {'No': 0, 'DSL': 1, 'Fiber optic': 2}
encoded_features['InternetService'] = encoded_features['InternetService'].map(internet_mapping)

nominal_columns = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
                   'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                   'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']
encoded_features = pd.get_dummies(encoded_features, columns=nominal_columns, drop_first=True)

for col in encoded_features.columns:
    if encoded_features[col].dtype == 'bool':
        encoded_features[col] = encoded_features[col].astype(int)
print("Encoded features shape: " + str(encoded_features.shape))

### Train-Test Split and Standardization

In [ ]:
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    encoded_features, target_regression, test_size=0.2, random_state=42
)
_, _, y_train_clf, y_test_clf = train_test_split(
    encoded_features, target_classification, test_size=0.2, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train size: " + str(X_train_scaled.shape[0]) + " | Test size: " + str(X_test_scaled.shape[0]))

### Linear vs. Ridge Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_reg)
pred_reg_lr = lr.predict(X_test_scaled)
print("OLS Test MSE: " + str(round(mean_squared_error(y_test_reg, pred_reg_lr), 4)) + 
      " | R2: " + str(round(r2_score(y_test_reg, pred_reg_lr), 4)))

print("Linear Regression Coefficients (Top 5):")
coef_df = pd.DataFrame({'Feature': encoded_features.columns, 'Coef': lr.coef_, 'Abs_Coef': np.abs(lr.coef_)})
print(coef_df.sort_values(by='Abs_Coef', ascending=False).head(5))

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_reg)
pred_reg_ridge = ridge.predict(X_test_scaled)
print("Ridge Test MSE: " + str(round(mean_squared_error(y_test_reg, pred_reg_ridge), 4)) + 
      " | R2: " + str(round(r2_score(y_test_reg, pred_reg_ridge), 4)))

### Classification Model (Logistic Regression)

In [ ]:
print("Training Logistic Regression with class weights...")
logistic_model = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42)
logistic_model.fit(X_train_scaled, y_train_clf)

predictions_clf = logistic_model.predict(X_test_scaled)
probabilities_clf = logistic_model.predict_proba(X_test_scaled)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test_clf, predictions_clf))
print("\nClassification Report:")
print(classification_report(y_test_clf, predictions_clf))
auc_score = roc_auc_score(y_test_clf, probabilities_clf)
print("ROC-AUC Score: " + str(round(auc_score, 4)))

In [ ]:
# Save ROC plot
fpr, tpr, thresholds = roc_curve(y_test_clf, probabilities_clf)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='blue', label='Logistic Regression (AUC = ' + str(round(auc_score, 4)) + ')')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.savefig("roc_curve.png" if os.path.exists("roc_curve.png") else "part2/roc_curve.png")
plt.show()

### Threshold Sensitivity Analysis

In [ ]:
threshold_list = [0.3, 0.4, 0.5, 0.6, 0.7]
print("Threshold | Precision | Recall | F1")
print("-----------------------------------")
for t in threshold_list:
    threshold_preds = [1 if p >= t else 0 for p in probabilities_clf]
    p = precision_score(y_test_clf, threshold_preds, zero_division=0)
    r = recall_score(y_test_clf, threshold_preds, zero_division=0)
    f = f1_score(y_test_clf, threshold_preds, zero_division=0)
    print(str(round(t, 2)) + "      | " + str(round(p, 4)) + "    | " + str(round(r, 4)) + " | " + str(round(f, 4)))

### Regularization Comparison & Bootstrap AUC

In [ ]:
regularized_model = LogisticRegression(class_weight='balanced', C=0.01, max_iter=1000, random_state=42)
regularized_model.fit(X_train_scaled, y_train_clf)
proba_reg_clf = regularized_model.predict_proba(X_test_scaled)[:, 1]

print("Regularization comparison:")
print("C=1.0 AUC: " + str(round(auc_score, 4)))
print("C=0.01 AUC: " + str(round(roc_auc_score(y_test_clf, proba_reg_clf), 4)))

In [ ]:
bootstrap_iterations = 500
differences = []
y_test_clf_array = np.array(y_test_clf)
proba_clf_array = np.array(probabilities_clf)
proba_reg_clf_array = np.array(proba_reg_clf)

np.random.seed(42)
for i in range(bootstrap_iterations):
    indices = np.random.choice(len(y_test_clf_array), size=len(y_test_clf_array), replace=True)
    sample_y = y_test_clf_array[indices]
    if len(np.unique(sample_y)) >= 2:
        auc_base = roc_auc_score(sample_y, proba_clf_array[indices])
        auc_reg = roc_auc_score(sample_y, proba_reg_clf_array[indices])
        differences.append(auc_base - auc_reg)

print("Bootstrap Mean AUC Diff: " + str(round(np.mean(differences), 6)))
print("95% CI: [" + str(round(np.percentile(differences, 2.5), 6)) + ", " + str(round(np.percentile(differences, 97.5), 6)) + "]")